In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
RAW_PATH = Path("../data/raw/online_retail_II.xlsx")

sheets = pd.read_excel(RAW_PATH, sheet_name=None, engine="openpyxl")
df = pd.concat(
    [sheet_df.assign(source_sheet=name) for name, sheet_df in sheets.items()],
    ignore_index=True,
)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]
print(df.shape)

(1067371, 9)


In [3]:
step_counts = {"raw": len(df)}

In [ ]:
# Cancellations: Invoice starting with "C" (the one "A" prefix is 6 rows with no
# Customer_ID, so it's removed later anyway, no special-casing needed)
df["Invoice"] = df["Invoice"].astype(str)
df_clean = df[~df["Invoice"].str.startswith("C")].copy()
step_counts["after_dropping_cancellations"] = len(df_clean)

In [5]:
# Non-positive quantity/price: confirmed these are returns/losses/adjustments, not sales
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Price"] > 0)]
step_counts["after_dropping_non_positive_qty_price"] = len(df_clean)

In [6]:
df_clean = df_clean.dropna(subset=["Customer_ID"])
df_clean["Customer_ID"] = df_clean["Customer_ID"].astype(int)
step_counts["after_dropping_missing_customer_id"] = len(df_clean)

In [7]:
# Non-product StockCodes, confirmed by inspecting Description for each one above.
# Codes like 15056BL, 79323LP/GR, PADS, SP1002 are real products with non-standard
# codes and are intentionally NOT excluded.
NON_PRODUCT_CODES = {
    "POST", "M", "C2", "ADJUST", "ADJUST2",
    "BANK CHARGES", "DOT", "TEST001", "TEST002", "D",
}
df_clean["StockCode"] = df_clean["StockCode"].astype(str)
df_clean = df_clean[~df_clean["StockCode"].str.upper().isin(NON_PRODUCT_CODES)]
step_counts["after_dropping_non_product_codes"] = len(df_clean)

In [8]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean = df_clean.drop_duplicates()
step_counts["after_dedup"] = len(df_clean)

summary = pd.DataFrame({
    "rows_remaining": step_counts,
    "pct_of_raw": {k: v / step_counts["raw"] * 100 for k, v in step_counts.items()},
})
summary

,rows_remaining,pct_of_raw
raw,1067371,100.000000
after_dropping_cancellations,1047877,98.173643
after_dropping_non_positive_qty_price,1041670,97.592121
after_dropping_missing_customer_id,805549,75.470385
after_dropping_non_product_codes,802651,75.198876
after_dedup,790721,74.081177


In [9]:
OUT_PATH = Path("../data/processed/transactions_clean.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(df_clean):,} rows to {OUT_PATH}")

Saved 790,721 rows to ..\data\processed\transactions_clean.parquet
